# CAP 5610 Midterm — Task 1
## Logistic Regression on the Adult Census Income Dataset

The goal is to predict whether an individual's annual income is greater
than $50,000. Logistic regression will be compared with and without class
weighting, establish a majority-class baseline, and evaluate the models
using accuracy, F1-score, and ROC-AUC.

## 1. Import Required Libraries

This section imports the libraries used for data handling, preprocessing,
model training, and evaluation.

- `numpy` and `pandas` are used for numerical operations and data handling.
- `matplotlib` is used for plots.
- `scikit-learn` provides preprocessing tools, logistic regression,
  pipelines, and evaluation metrics.

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

#For Missing Values
from sklearn.impute import SimpleImputer
#For Data Pre-Processing
from sklearn.compose import ColumnTransformer

#For Logistic Regression
from sklearn.linear_model import LogisticRegression

#Stastical Scores Required for this task
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    confusion_matrix
)

#Importing pipeline to combine preprocessing and modelling into one workflow
from sklearn.pipeline import Pipeline

#Importing One-Hot encoder to convert categorical variables to numerical values and Standard Scaler to standardize continuous
#numerical variables
from sklearn.preprocessing import StandardScaler,OneHotEncoder

print("All Imports Successful")

All Imports Successful


## 2. Load the Adult Census Training and Test Data

-> We have been provided separate training and test datasets.

-> The training set will be used to fit the preprocessing steps and the
Logistic Regression model. The test set will only be used after training
to evaluate how well the model generalizes to unseen observations.

In [2]:
# Paths are relative to this notebook.
# The notebook is inside the "notebooks" folder, so "../data/"
# moves up one level and then enters the "data" folder.

train_path = "../data/adult_train_.csv"
test_path = "../data/adult_test_.csv"

# Load the supplied CSV files into pandas DataFrames.
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

## 3. Initial Inspection of the Dataset

Before preprocessing or training a model, we first inspect the size,
column names, data types, and a few example rows.

This helps verify that the files were loaded correctly and also helps
identify which variables are numerical and which are categorical.

In [3]:
print("Training Shape:", train_df.shape)
print("Testing Shape:", test_df.shape)

train_df.head()

Training Shape: (26048, 14)
Testing Shape: (6513, 14)


,age,workclass,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,label->50K
0,33,Local-gov,Bachelors,13,Never-married,Prof-specialty,Not-in-family,White,Female,0,0,50,United-States,1
1,36,Private,Assoc-voc,11,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,1887,50,United-States,1
2,58,Self-emp-not-inc,9th,5,Separated,Craft-repair,Not-in-family,White,Male,0,0,40,United-States,0
3,21,Private,Assoc-voc,11,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,46,United-States,0
4,27,Private,Some-college,10,Divorced,Other-service,Not-in-family,White,Male,0,0,40,United-States,0


In [4]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26048 entries, 0 to 26047
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             26048 non-null  int64
 1   workclass       26048 non-null  str  
 2   education       26048 non-null  str  
 3   education_num   26048 non-null  int64
 4   marital_status  26048 non-null  str  
 5   occupation      26048 non-null  str  
 6   relationship    26048 non-null  str  
 7   race            26048 non-null  str  
 8   sex             26048 non-null  str  
 9   capital_gain    26048 non-null  int64
 10  capital_loss    26048 non-null  int64
 11  hours_per_week  26048 non-null  int64
 12  native_country  26048 non-null  str  
 13  label->50K      26048 non-null  int64
dtypes: int64(6), str(8)
memory usage: 2.8 MB


## 4. Inspect Columns and Data Types

The Adult dataset contains both continuous and categorical variables.
Before the preprocessing pipeline is created, the column names and data
types are inspected so that an appropriate transformation can be applied
to each feature.

The target variable is `label->50K`.

In [5]:
#Display all columns as they appear
print("Columns:")
print(train_df.columns.tolist())

#Display data type for each column
print("Data Type:")
print(train_df.dtypes)

Columns:
['age', 'workclass', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'label->50K']
Data Type:
age               int64
workclass           str
education           str
education_num     int64
marital_status      str
occupation          str
relationship        str
race                str
sex                 str
capital_gain      int64
capital_loss      int64
hours_per_week    int64
native_country      str
label->50K        int64
dtype: object


## 5. Check for Missing Values

Missing values must be handled before any processing may begin. Following steps are needed:
1. The dataset is first checked for standard missing values such as `NaN`.
2. However, the Adult dataset may also use the string `"?"` to represent
missing categorical values. Therefore, both forms are checked.

In [6]:
print("Standard Missing Values in the Train Dataset:")
print(train_df.isna().sum())

#Doing the same for test set as well
print("\nStandard missing values in Test Dataset:")
print(test_df.isna().sum())

Standard Missing Values in the Train Dataset:
age               0
workclass         0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
label->50K        0
dtype: int64

Standard missing values in Test Dataset:
age               0
workclass         0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
label->50K        0
dtype: int64


In [7]:
#Check for `"?"` Missing-Value Markers - not automatically done by pandas methods. Needs to be checked explicitly
question_marks_train = (train_df == "?").sum()
question_marks_test = (test_df == "?").sum()

# Display only columns that contain "?".
print(" '?' values in Train Data:")
print(question_marks_train[question_marks_train > 0])

print("\n'?' values in Test Data:")
print(question_marks_test[question_marks_test > 0])


#Replace the ? with numpy NaN
train_df = train_df.replace("?",np.nan)
test_df = test_df.replace("?", np.nan)

#Checking if missing values are being identified
print("Missing Values after conversion in Train Data:")
print(train_df.isna().sum()[train_df.isna().sum()>0])

print("\n Missing values after conversion in Test Data:")
print(test_df.isna().sum()[test_df.isna().sum()>0])

 '?' values in Train Data:
workclass         1447
occupation        1454
native_country     458
dtype: int64

'?' values in Test Data:
workclass         389
occupation        389
native_country    125
dtype: int64
Missing Values after conversion in Train Data:
workclass         1447
occupation        1454
native_country     458
dtype: int64

 Missing values after conversion in Test Data:
workclass         389
occupation        389
native_country    125
dtype: int64


## 6. Inspect Class Distribution

The target variable is `label->50K`, where:

- `0` represents annual income less than or equal to \$50,000.
- `1` represents annual income greater than \$50,000.

Because we know that the dataset is imbalanced, the class
counts and proportions are examined before model training.

In [8]:
#Counting number of observations belonging to each target class.
class_counts = (
    train_df["label->50K"].value_counts()
    .sort_index()
)

#Counting proportions
class_proportions = (
    train_df["label->50K"]
    .value_counts(normalize=True)
    .sort_index()
)

print("Training Class Counts:")
print(class_counts)

print("\nTraining Class Proportions:")
print(class_proportions)


Training Class Counts:
label->50K
0    19778
1     6270
Name: count, dtype: int64

Training Class Proportions:
label->50K
0    0.759291
1    0.240709
Name: proportion, dtype: float64


## 7. Majority-Class Baseline

Before Logistic Regression is evaluated, a naive majority-class baseline
is established.

This baseline ignores all predictor variables and assigns the majority
class to every observation in the test set. It provides a reference
showing how much accuracy can be obtained without learning any
relationship between the features and the target.

The trained model should perform meaningfully better than this baseline.

In [9]:
# Extract the test target values.
y_test = test_df["label->50K"]

# Identify the majority class using the training set only.
# This avoids using information from the test set when establishing
# the baseline prediction rule.
majority_class = train_df["label->50K"].mode()[0]

# Predict the majority class for every observation in the test set.
baseline_predictions = np.full(
    shape=len(y_test),
    fill_value=majority_class
)

# Calculate the test accuracy of the naive baseline.
baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Majority class:", majority_class)
print(f"Majority-class baseline accuracy: {baseline_accuracy:.4f}")
print(f"Majority-class baseline accuracy: {baseline_accuracy * 100:.2f}%")
print("Baseline F1:", f1_score(y_test, baseline_predictions))          # 0, never predicts 1
print("Baseline ROC-AUC:", roc_auc_score(y_test, baseline_predictions))  # 0.5

Majority class: 0
Majority-class baseline accuracy: 0.7588
Majority-class baseline accuracy: 75.88%
Baseline F1: 0.0
Baseline ROC-AUC: 0.5


## 8. Separate Predictors and Target

Before preprocessing and model training, the target variable is separated
from the predictor variables.

The predictor variables are stored in `X_train` and `X_test`, while the
target values are stored in `y_train` and `y_test`.

The test set remains separate from model fitting so that it can later be
used to evaluate performance on unseen observations.

In [10]:
#Removing target column and training data and repeating the same steps on test data
X_train = train_df.drop(columns=["label->50K"])
y_train = train_df["label->50K"]

X_test = test_df.drop(columns=["label->50K"])
y_test = test_df["label->50K"]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (26048, 13)
y_train shape: (26048,)
X_test shape: (6513, 13)
y_test shape: (6513,)


## 9. Identify Numerical and Categorical Features

The Adult dataset contains both numerical and categorical predictors, and
the two feature types require different preprocessing steps.

The continuous variables specified in the assignment are `age`,
`education_num`, and `hours_per_week`.

The remaining predictors are treated as categorical. Although
`capital_gain` and `capital_loss` are stored as numerical values in the
CSV files, they are treated as categorical variables according to the
assignment specification. The supplied dataset also contains
`occupation`, which is treated as categorical.

In [11]:
# Continuous features specified in the assignment.
numerical_features = [
    "age",
    "education_num",
    "hours_per_week"
]

# Categorical features specified in the assignment.
# occupation is also included because it is present in the supplied dataset.
categorical_features = [
    "workclass",
    "education",
    "marital_status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital_gain",
    "capital_loss",
    "native_country"
]

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['age', 'education_num', 'hours_per_week']

Categorical Features:
['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'native_country']


## 10. Build the Preprocessing Pipelines

Different preprocessing steps are applied to numerical and categorical
features.

For numerical variables:

1. Missing values are replaced using the median.
2. The values are standardized using `StandardScaler`.

For categorical variables:

1. Missing values are replaced using the most frequent category.
2. The categories are converted into numerical indicator variables using
   one-hot encoding.

These preprocessing steps are fitted only on the training data and are
later applied to the test data.

In [12]:
#Working on Numerical Variables
numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

## 11. Combine Numerical and Categorical Preprocessing

A `ColumnTransformer` is used to apply different preprocessing steps to
different groups of columns.

The numerical pipeline is applied only to the continuous predictors,
while the categorical pipeline is applied only to the categorical
predictors.

This produces one fully numerical feature matrix that can be passed to
Logistic Regression.

In [13]:
#Combining Both Pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ("num",numerical_pipeline,numerical_features),
        ("cat",categorical_pipeline,categorical_features)
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'education_num', 'hours_per_week']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['workclass', 'education', 'marital_status',
                                  'occupation', 'relationship', 'race', 'sex',
                                  'capital_gain', 'capital_loss',
                                  'native_country'])])


## 12. Train Logistic Regression with Balanced Class Weights

The first Logistic Regression model uses `class_weight="balanced"` as
required by the assignment.

Because the high-income class is less common, balanced class weights give
greater importance to errors involving the minority class during model
training.

The preprocessing steps and Logistic Regression model are combined into
a single pipeline so that all transformations are fitted using only the
training data.

In [14]:
#Logistic Regression with Balanced Class Weights
balanced_logistic = LogisticRegression(
    class_weight="balanced",
    C=1.0,
    max_iter = 1000,
    random_state = 42
)

#Combining preprocessing and Logistic Regression into one pipeline
balanced_model = Pipeline(
    steps=[
        ("preprocessor",clone(preprocessor)),
        ("classifier",balanced_logistic)
    ]
)

### Effect of Balanced Class Weights

The training set contains substantially more observations from class `0`
than class `1`. If both classes are treated equally during optimization,
the model may favor the majority class because errors on that class
dominate the loss.

Using `class_weight="balanced"` automatically assigns a larger weight to
the minority class and a smaller weight to the majority class. The class
weight is approximately

\[
w_c = \frac{n}{k n_c}
\]

where \(n\) is the total number of training observations, \(k\) is the
number of classes, and \(n_c\) is the number of training observations in
class \(c\).

This increases the penalty for misclassifying the minority high-income
class and encourages the model to pay more attention to those
observations.

## 13. Fit the Balanced Logistic Regression Model

The complete preprocessing and Logistic Regression pipeline is fitted on
the training data.

The preprocessing steps learn their parameters from the training set
only. The fitted transformations are then passed directly into the
Logistic Regression classifier.

In [15]:
# Fit the complete pipeline on the training data.
balanced_model.fit(X_train, y_train)

print("Balanced Logistic Regression training complete.")

Balanced Logistic Regression training complete.


## 14. Generate Test Predictions

Two forms of predictions are generated:

- Class predictions (`0` or `1`) are used for accuracy and F1-score.
- Predicted probabilities for class `1` are used for ROC-AUC.

ROC-AUC is calculated from probabilities rather than hard class labels
because it measures how well the model ranks positive observations
across different classification thresholds.

In [16]:
# Predict the final class label for each test observation.
balanced_predictions = balanced_model.predict(X_test)

# Predict the probability that each observation belongs to class 1.
# [:, 1] selects the probability associated with the positive class.
balanced_probabilities = balanced_model.predict_proba(X_test)[:, 1]

## 15. Evaluate the Balanced Model

The balanced Logistic Regression model is evaluated using the three
metrics required:

- Accuracy
- F1-score
- ROC-AUC

'Accuracy measures the overall proportion of correct predictions.'

'F1-score focuses more strongly on performance for the positive class and
combines precision and recall.'

'ROC-AUC evaluates how well the model separates the two classes across
different classification thresholds.'

In [17]:
# Calculate the required evaluation metrics.
balanced_accuracy = accuracy_score(
    y_test,
    balanced_predictions
)

balanced_f1 = f1_score(
    y_test,
    balanced_predictions
)

balanced_auc = roc_auc_score(
    y_test,
    balanced_probabilities
)

print(f"Balanced Accuracy: {balanced_accuracy:.4f}")
print(f"Balanced F1-score: {balanced_f1:.4f}")
print(f"Balanced ROC-AUC: {balanced_auc:.4f}")

Balanced Accuracy: 0.8248
Balanced F1-score: 0.7015
Balanced ROC-AUC: 0.9226


### Balanced Model Results

The balanced Logistic Regression model achieved:

- Accuracy: **0.8248**
- F1-score: **0.7015**
- ROC-AUC: **0.9226**

The model clearly outperforms the majority-class baseline accuracy of
approximately 75.9%. More importantly, the F1-score of 0.7015 shows that
the model is substantially better at identifying the minority high-income
class than the majority-class baseline, which never predicts class `1`.

The ROC-AUC of 0.9226 indicates strong discrimination between the two
income classes across classification thresholds.

## 16. Extract and Interpret Logistic Regression Coefficients

Logistic Regression represents the log-odds of the positive class as


$$
p(x) = P(Y=1 \mid X=x).
$$

Logistic Regression models the log-odds of the positive class as

$$
\log\left(\frac{p(x)}{1-p(x)}\right)
=
\beta_0 + \sum_{j=1}^{p}\beta_j x_j.
$$

A positive coefficient increases the log-odds that an observation belongs
to the high-income class, while a negative coefficient decreases those
log-odds.

Because preprocessing creates additional one-hot encoded variables, the
feature names produced by the preprocessing pipeline are first recovered
and matched with the fitted coefficients.

In [18]:
# Retrieve the fitted preprocessing step from the full model pipeline.
fitted_preprocessor = balanced_model.named_steps["preprocessor"]

# Obtain the names of all features after preprocessing.
# This includes standardized numerical features and one-hot encoded
# categorical features.
feature_names = fitted_preprocessor.get_feature_names_out()

# Retrieve the fitted Logistic Regression classifier.
fitted_classifier = balanced_model.named_steps["classifier"]

# Logistic Regression stores one coefficient for each transformed feature.
# For binary classification, coef_[0] contains that coefficient vector.
coefficients = fitted_classifier.coef_[0]

# Store feature names and their corresponding beta coefficients together.
coefficient_df = pd.DataFrame({
    "feature": feature_names,
    "beta": coefficients
})

# Remove the ColumnTransformer prefixes to make the names easier to read.
coefficient_df["feature"] = (
    coefficient_df["feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

print("Number of transformed features:", len(coefficient_df))

coefficient_df.head()

Number of transformed features: 304


,feature,beta
0,age,0.373163
1,education_num,0.748041
2,hours_per_week,0.427095
3,workclass_Federal-gov,0.721635
4,workclass_Local-gov,0.070088


In [19]:
# Identify the features with the largest positive coefficients.
largest_positive = (
    coefficient_df
    .sort_values("beta", ascending=False)
    .head(10)
)

# Identify the features with the most negative coefficients.
largest_negative = (
    coefficient_df
    .sort_values("beta", ascending=True)
    .head(10)
)

print("Largest positive coefficients:")
print(largest_positive.to_string(index=False))

print("\nLargest negative coefficients:")
print(largest_negative.to_string(index=False))

Largest positive coefficients:
           feature     beta
capital_gain_99999 4.458975
 capital_gain_8614 4.347839
capital_gain_14344 4.157853
capital_gain_10520 3.961513
 capital_loss_1564 3.911934
 capital_gain_7298 3.714087
capital_gain_27828 3.689581
 capital_gain_7688 3.657710
capital_gain_14084 3.614146
capital_gain_13550 3.578656

Largest negative coefficients:
          feature      beta
capital_gain_5013 -3.643896
capital_gain_4064 -3.213050
capital_loss_1740 -3.078100
capital_gain_3908 -2.822125
capital_gain_3137 -2.699994
capital_loss_1672 -2.681178
capital_gain_2829 -2.622633
capital_gain_2885 -2.569397
capital_gain_2407 -2.566172
capital_loss_2002 -2.369153


In [20]:
# Examine the strongest coefficients outside the sparse
# capital-gain and capital-loss categories.
non_capital_coefficients = coefficient_df[
    ~coefficient_df["feature"].str.startswith(
        ("capital_gain_", "capital_loss_")
    )
]

largest_positive_noncapital = (
    non_capital_coefficients
    .sort_values("beta", ascending=False)
    .head(10)
)

largest_negative_noncapital = (
    non_capital_coefficients
    .sort_values("beta", ascending=True)
    .head(10)
)

print("Largest positive non-capital coefficients:")
print(largest_positive_noncapital.to_string(index=False))

print("\nLargest negative non-capital coefficients:")
print(largest_negative_noncapital.to_string(index=False))

Largest positive non-capital coefficients:
                          feature     beta
 marital_status_Married-AF-spouse 1.694741
marital_status_Married-civ-spouse 1.488179
          native_country_Cambodia 1.317403
                relationship_Wife 1.166237
             native_country_Italy 1.029816
       occupation_Exec-managerial 0.841059
        native_country_Yugoslavia 0.839052
                    education_num 0.748041
            workclass_Federal-gov 0.721635
          occupation_Tech-support 0.709211

Largest negative non-capital coefficients:
                                  feature      beta
               occupation_Priv-house-serv -1.470895
                  native_country_Columbia -1.394997
             marital_status_Never-married -1.153186
                    workclass_Without-pay -1.113519
                   relationship_Own-child -1.078003
                      native_country_Peru -1.027500
               occupation_Farming-fishing -1.026234
                 marital

The overall largest coefficients are dominated by individual
`capital_gain` and `capital_loss` categories. Since some exact monetary
values may occur infrequently, these coefficients can become large and
should be interpreted cautiously.

For a more substantive interpretation, the strongest coefficients among
the remaining predictors are also examined. Positive coefficients increase
the log-odds of income above $50,000, while negative coefficients decrease
them, holding the other model inputs fixed.

Because all one-hot encoded levels are retained, there is no single
reference category. However, two levels of the same categorical variable
can be compared using

$$
\exp(\beta_a-\beta_b),
$$

which gives the multiplicative change in odds between those two levels,
holding the other predictors fixed.

### Example Coefficient Interpretations

For the standardized numerical feature `education_num`, the fitted
coefficient is approximately

$$
\beta = 0.748.
$$

Because the feature is standardized,

$$
e^{0.748} \approx 2.11.
$$

Thus, holding the other predictors fixed, an increase of one standard
deviation in `education_num` multiplies the predicted odds of earning
more than $50,000 by approximately **2.11**.

Categorical levels of the same variable can be compared by taking the
difference between their coefficients. For example,

- `Married-civ-spouse`: $\beta \approx 1.488$
- `Never-married`: $\beta \approx -1.153$

Therefore,

$$
e^{1.488 - (-1.153)}
=
e^{2.641}
\approx 14.0.
$$

Holding the other predictors fixed, the fitted model assigns the
`Married-civ-spouse` category approximately **14 times the odds** of the
`Never-married` category for income above $50,000.

## 17. Train Logistic Regression Without Balanced Class Weights

To examine the effect of class weighting, a second Logistic Regression
model is trained using the same preprocessing steps but without
`class_weight="balanced"`.

This allows the performance of the weighted and unweighted models to be
compared directly.

In [21]:
# Create Logistic Regression without balanced class weights.
unbalanced_logistic = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

# Use exactly the same preprocessing pipeline.
unbalanced_model = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", unbalanced_logistic)
    ]
)

# Fit the model on the training data.
unbalanced_model.fit(X_train, y_train)

print("Unbalanced Logistic Regression training complete.")

Unbalanced Logistic Regression training complete.


In [22]:
# Generate class predictions.
unbalanced_predictions = unbalanced_model.predict(X_test)

# Generate probabilities for the positive class.
unbalanced_probabilities = unbalanced_model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics.
unbalanced_accuracy = accuracy_score(
    y_test,
    unbalanced_predictions
)

unbalanced_f1 = f1_score(
    y_test,
    unbalanced_predictions
)

unbalanced_auc = roc_auc_score(
    y_test,
    unbalanced_probabilities
)

print(f"Unbalanced Accuracy: {unbalanced_accuracy:.4f}")
print(f"Unbalanced F1-score: {unbalanced_f1:.4f}")
print(f"Unbalanced ROC-AUC: {unbalanced_auc:.4f}")

Unbalanced Accuracy: 0.8695
Unbalanced F1-score: 0.6996
Unbalanced ROC-AUC: 0.9219


In [23]:
balanced_precision = precision_score(
    y_test,
    balanced_predictions
)

balanced_recall = recall_score(
    y_test,
    balanced_predictions
)

unbalanced_precision = precision_score(
    y_test,
    unbalanced_predictions
)

unbalanced_recall = recall_score(
    y_test,
    unbalanced_predictions
)

print("Balanced model")
print(f"Precision: {balanced_precision:.4f}")
print(f"Recall:    {balanced_recall:.4f}")
print("Confusion matrix:")
print(confusion_matrix(y_test, balanced_predictions))

print("\nUnbalanced model")
print(f"Precision: {unbalanced_precision:.4f}")
print(f"Recall:    {unbalanced_recall:.4f}")
print("Confusion matrix:")
print(confusion_matrix(y_test, unbalanced_predictions))

Balanced model
Precision: 0.5955
Recall:    0.8536
Confusion matrix:
[[4031  911]
 [ 230 1341]]

Unbalanced model
Precision: 0.7863
Recall:    0.6302
Confusion matrix:
[[4673  269]
 [ 581  990]]


In [24]:
# Retrieve the fitted unbalanced Logistic Regression classifier.
unbalanced_classifier = unbalanced_model.named_steps["classifier"]

# Retrieve the transformed feature names from its preprocessing pipeline.
unbalanced_feature_names = (
    unbalanced_model
    .named_steps["preprocessor"]
    .get_feature_names_out()
)

# Retrieve the beta coefficients.
unbalanced_coefficients = unbalanced_classifier.coef_[0]

# Store feature names and coefficients together.
unbalanced_coefficient_df = pd.DataFrame({
    "feature": unbalanced_feature_names,
    "beta": unbalanced_coefficients
})

# Remove preprocessing prefixes for readability.
unbalanced_coefficient_df["feature"] = (
    unbalanced_coefficient_df["feature"]
    .str.replace("num__", "", regex=False)
    .str.replace("cat__", "", regex=False)
)

# Display the largest positive and negative coefficients.
print("Largest positive coefficients — unbalanced model:")
print(
    unbalanced_coefficient_df
    .sort_values("beta", ascending=False)
    .head(10)
    .to_string(index=False)
)

print("\nLargest negative coefficients — unbalanced model:")
print(
    unbalanced_coefficient_df
    .sort_values("beta", ascending=True)
    .head(10)
    .to_string(index=False)
)

Largest positive coefficients — unbalanced model:
           feature     beta
 capital_gain_8614 4.258858
capital_gain_99999 4.189210
capital_gain_14344 3.853312
capital_gain_10520 3.830606
 capital_loss_1564 3.800873
 capital_gain_7298 3.695740
 capital_gain_7688 3.651491
 capital_loss_1887 3.641524
capital_gain_14084 3.470240
capital_gain_13550 3.411599

Largest negative coefficients — unbalanced model:
          feature      beta
capital_gain_5013 -3.282053
capital_gain_4064 -2.930121
capital_loss_1740 -2.723659
capital_gain_3908 -2.551105
capital_gain_2829 -2.404634
capital_gain_3137 -2.404462
capital_gain_2407 -2.388856
capital_gain_2885 -2.348367
capital_loss_1672 -2.336739
capital_gain_3464 -2.172649


In [25]:
# Compare the majority baseline, balanced model, and unbalanced model.
comparison_df = pd.DataFrame({
    "Model": [
        "Majority Baseline",
        "Balanced Logistic Regression",
        "Unbalanced Logistic Regression"
    ],
    "Accuracy": [
        baseline_accuracy,
        balanced_accuracy,
        unbalanced_accuracy
    ],
    "F1-score": [
        0.0,
        balanced_f1,
        unbalanced_f1
    ],
    "ROC-AUC": [
        0.5,
        balanced_auc,
        unbalanced_auc
    ]
})

# Round the displayed values for readability.
comparison_df.round(4)

,Model,Accuracy,F1-score,ROC-AUC
0,Majority Baseline,0.7588,0.0000,0.5000
1,Balanced Logistic Regression,0.8248,0.7015,0.9226
2,Unbalanced Logistic Regression,0.8695,0.6996,0.9219


### Effect of Class Weighting

The unbalanced Logistic Regression model achieved higher overall accuracy
(**0.8695**) than the balanced model (**0.8248**). The two models were
essentially tied on F1-score (**0.7015** versus **0.6996**) and ROC-AUC
(**0.9226** versus **0.9219**). The main difference between the models is
therefore not their overall discrimination, but the trade-off between
precision and recall.

The confusion matrices show the trade-off more clearly. The balanced model
correctly identified **1,341** high-income observations, compared with
**990** for the unbalanced model. Therefore, balancing allowed the model
to correctly identify **351 additional high-income observations**.
However, this came at the cost of substantially more false positives:
**911** for the balanced model compared with **269** for the unbalanced
model, an increase of **642 false alarms**.

This explains the lower accuracy of the balanced model. It made
**1,141 total classification errors** (911 false positives + 230 false
negatives), while the unbalanced model made **850 errors** (269 false
positives + 581 false negatives).

The F1-scores remain very similar because class weighting changes the
precision-recall trade-off in opposite directions. The balanced model has
much higher recall (**0.8536**) but lower precision (**0.5955**), while the
unbalanced model has lower recall (**0.6302**) but higher precision
(**0.7863**). Since F1 combines precision and recall, these opposing
changes produce similar F1 values.

The ROC-AUC values are also nearly identical. This indicates that the two
models have very similar ability to rank high-income observations above
low-income observations across classification thresholds, even though
their predictions differ at the default threshold.

The coefficient comparison supports the same conclusion. The largest
positive and negative coefficients in both models are dominated by many
of the same `capital_gain` and `capital_loss` categories, with the same
general coefficient directions. Class weighting therefore changes the
fitted decision function and the relative magnitudes of some coefficients,
but does not fundamentally change the main patterns learned by the model.

The preferred model depends on the cost of the two types of errors. If
missing a high-income observation is more costly, the balanced model would
be preferred because of its substantially higher recall. If false positive
predictions are more costly, the unbalanced model would be preferred
because of its much higher precision and lower overall error rate.